In [26]:
!apt-get update -qq && apt-get install -y maven

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
maven is already the newest version (3.6.3-5).
0 upgraded, 0 newly installed, 0 to remove and 90 not upgraded.


In [27]:
import zipfile
import os

zip_path = "/content/javacpp.zip"
extract_path = "/content/javacpp"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for file in files:
        print("  " * (level + 1) + file)

javacpp/
  javacpp/
    VectorExample.java
    vectorlib.cpp
    vectorlib.lib
    jcuda/
      .gitignore
      pom.xml
      libvectorlib.so
      vectorlib.cu
      target/
        jcuda-1.0-SNAPSHOT.jar
        classes/
          VectorExample.class
          Main.class
        maven-archiver/
          pom.properties
        maven-status/
          maven-compiler-plugin/
            testCompile/
              default-testCompile/
                createdFiles.lst
                inputFiles.lst
            compile/
              default-compile/
                createdFiles.lst
                inputFiles.lst
        generated-test-sources/
          test-annotations/
        generated-sources/
          annotations/
        test-classes/
          JavaCppTest.class
      src/
        test/
          java/
            JavaCppTest.java
            com/
              gabriel/
        main/
          java/
            Main.java
            VectorExample.java
          resources/
      .

In [28]:
%%writefile /content/javacpp/javacpp/jcuda/src/main/java/VectorExample.java
import java.util.Arrays;

public class VectorExample {
    static {
        System.loadLibrary("vectorlib");
    }

    public native int[] addVectors(int[] a, int[] b);

    public static void main(String[] args) {
        int[] a = {1, 2, 3, 4, 5};
        int[] b = {10, 20, 30, 40, 50};

        VectorExample example = new VectorExample();
        int[] result = example.addVectors(a, b);

        System.out.println("Vector A: " + Arrays.toString(a));
        System.out.println("Vector B: " + Arrays.toString(b));
        System.out.println("Result:   " + Arrays.toString(result));
    }
}

Overwriting /content/javacpp/javacpp/jcuda/src/main/java/VectorExample.java


In [29]:
%%writefile /content/javacpp/javacpp/jcuda/vectorlib.cu
#include <jni.h>
#include <cuda_runtime.h>

__global__ void addKernel(const int* a, const int* b, int* c, int size) {
    int index = blockIdx.x * blockDim.x + threadIdx.x;

    if (index < size) {
        c[index] = a[index] + b[index];
    }
}

extern "C" JNIEXPORT jintArray JNICALL
Java_VectorExample_addVectors(JNIEnv* env, jobject obj, jintArray arrA, jintArray arrB) {
    int size = env->GetArrayLength(arrA);

    jint* a = env->GetIntArrayElements(arrA, nullptr);
    jint* b = env->GetIntArrayElements(arrB, nullptr);

    int *deviceA, *deviceB, *deviceC;
    int bytes = size * sizeof(int);

    cudaMalloc(&deviceA, bytes);
    cudaMalloc(&deviceB, bytes);
    cudaMalloc(&deviceC, bytes);

    cudaMemcpy(deviceA, a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(deviceB, b, bytes, cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (size + threads - 1) / threads;

    addKernel<<<blocks, threads>>>(deviceA, deviceB, deviceC, size);
    cudaDeviceSynchronize();

    int* result = new int[size];
    cudaMemcpy(result, deviceC, bytes, cudaMemcpyDeviceToHost);

    jintArray output = env->NewIntArray(size);
    env->SetIntArrayRegion(output, 0, size, result);

    cudaFree(deviceA);
    cudaFree(deviceB);
    cudaFree(deviceC);

    delete[] result;

    env->ReleaseIntArrayElements(arrA, a, JNI_ABORT);
    env->ReleaseIntArrayElements(arrB, b, JNI_ABORT);

    return output;
}

Overwriting /content/javacpp/javacpp/jcuda/vectorlib.cu


In [30]:
!rm -f /content/javacpp/javacpp/jcuda/src/main/java/Main.java

In [31]:
%cd /content/javacpp/javacpp/jcuda
!mvn -Dmaven.test.skip=true clean package

/content/javacpp/javacpp/jcuda
[INFO] Scanning for projects...
[INFO] 
[INFO] ---------------------< com.gabriel.javacpp:jcuda >----------------------
[INFO] Building jcuda 1.0-SNAPSHOT
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] --- maven-clean-plugin:2.5:clean (default-clean) @ jcuda ---
[INFO] Deleting /content/javacpp/javacpp/jcuda/target
[INFO] 
[INFO] --- maven-resources-plugin:2.6:resources (default-resources) @ jcuda ---
[INFO] Using 'UTF-8' encoding to copy filtered resources.
[INFO] Copying 0 resource
[INFO] 
[INFO] --- maven-compiler-plugin:3.1:compile (default-compile) @ jcuda ---
[INFO] Changes detected - recompiling the module!
[INFO] Compiling 1 source file to /content/javacpp/javacpp/jcuda/target/classes
[INFO] 
[INFO] --- maven-resources-plugin:2.6:testResources (default-testResources) @ jcuda ---
[INFO] Not copying test resources
[INFO] 
[INFO] --- maven-compiler-plugin:3.1:testCompile (default-testCompile) @ jcuda --

In [32]:
!java -Djava.library.path=. -cp target/classes VectorExample

Vector A: [1, 2, 3, 4, 5]
Vector B: [10, 20, 30, 40, 50]
Result:   [11, 22, 33, 44, 55]
